# Parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/02_parameters.ipynb)

Every number in minilink is a parameter: one tree per object, passed as an argument, viewed live through diagrams and problems.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Every number is a parameter

A model's numbers live in one dictionary, `params`, never in attributes the equations read
behind your back. Every equation method takes it as its last argument: `f(x, u, t, params)`
reads the stored dictionary when `params` is `None`, and the tree you pass otherwise. That
one convention is what lets a tool evaluate, sweep, randomize or differentiate a model
without touching the object.

In [ ]:
import copy

import numpy as np

from minilink import Pendulum

plant = Pendulum()
plant.params

In [ ]:
x = np.array([0.5, 0.0])
u = np.array([0.0])

heavy = {**plant.params, "m": 2.0, "d": 0.5}
plant.f(x, u), plant.f(x, u, params=heavy)

Because the parameters are an argument, they are also a variable of differentiation. The
sensitivity $\partial f / \partial p$ comes per parameter, by automatic differentiation
when JAX is installed:

In [ ]:
plant.jacobian("f", "params", x, u)

## 2. A diagram's tree is a live view of its blocks

A diagram's `params` is a nested dictionary keyed by block id (`"controller"`, `"plant"`,
`"error"`, ... as the operators name them). It is a view, not a copy: writing a leaf through
the diagram writes the block's own dictionary. A tree passed to the diagram's `f` has the
same nested shape.

In [ ]:
from minilink import Impedance

loop = Impedance() @ plant
loop.params

In [ ]:
loop.params["plant"]["m"] = 1.5
plant.params["m"]

In [ ]:
stiff = copy.deepcopy(loop.params)
stiff["controller"]["Kp"] = np.array([[50.0]])

r = np.array([0.0])
loop.f(x, r), loop.f(x, r, params=stiff)

## 3. Sets, fields and distributions hold their numbers the same way

The other mathematical objects follow the same rule. A `BoxSet` holds `lower` and `upper`,
a `QuadraticField` holds `center` and `P`, a `Gaussian` holds `mean` and `std`. Their
equation methods (`margin`, `value`, `sample`) take a `params` override, and `with_params`
returns a copy holding new numbers while the original stays as it was.

In [ ]:
from minilink import BoxSet, Gaussian

box = BoxSet([-1.0, -1.0], [1.0, 1.0])
small_box = box.with_params(
    {"lower": np.array([-0.2, -0.2]), "upper": np.array([0.2, 0.2])}
)

box.params, box.contains(x), small_box.contains(x)

In [ ]:
start_law = Gaussian(np.zeros(2), std=0.1)
shifted_law = start_law.with_params(
    {"mean": np.array([0.5, 0.0]), "std": np.array([0.1, 0.1])}
)

start_law.mean, shifted_law.mean, shifted_law.sample(0, 3)

## 4. Costs hold parameters too

A cost's weights and targets are its `params` (`Q`, `R`, `xbar`, ... for a quadratic cost).
Its `g` and `h` take the same override. A cost has no `with_params`: to change its stored
numbers, assign a new dictionary to `cost.params`.

In [ ]:
from minilink import QuadraticCost

cost = QuadraticCost.from_system(plant, Q=np.eye(2), R=np.eye(1))
g_nominal = cost.g(x, u, 0.0)

cost.params = {**cost.params, "Q": 10.0 * np.eye(2)}
g_heavier = cost.g(x, u, 0.0)
g_at_target = cost.g(x, u, 0.0, params={**cost.params, "xbar": x})

g_nominal, g_heavier, g_at_target

## 5. The problem's tree: every number, and a law where a number goes

A `PlanningProblem` gathers the system, the sets, the cost and the horizon. Its `params` is
the live view of all their numbers, by role: `plant`, `cost`, `X`, `U`, `x_start`,
`t_start`, `discount_rate`. Where a number goes, a law may go: a `Distribution` as the
start, or laws over the plant's parameters in the tree's own shape. `realize(key, n)` draws
`n` realizations into one tree, each drawn leaf with a leading trial axis.

In [ ]:
from minilink import PlanningProblem, Uniform

problem = PlanningProblem(
    plant,
    x_start=Gaussian(np.zeros(2), std=0.1),
    cost=cost,
    tf=5.0,
    params={"plant": {"m": Uniform([1.2], [1.8])}},
)
list(problem.params), problem.params["plant"]["m"]

In [ ]:
draws = problem.realize(0, 3)
draws["plant"]["m"], draws["x_start"]

When the problem is compiled with a controller (see `06_compile_and_evaluate`), the
evaluator's tree adds the controller's numbers next to these. That single tree is what
`jax.grad` differentiates and what an optimizer moves.

Next, [`03_sets_fields_distributions`](03_sets_fields_distributions.ipynb): the objects
behind `X`, `U`, `X0`, `Xf` and the laws.